# Pokémon Gen 2 OU — Three reproducible submissions

The notebook generates the observed-offense, full baseline and unanimous-expert submissions from one training run. All features come from the competition's official files. Reported CV scores are exploratory and not guarantees of public Kaggle performance.

## Clone the repository and install dependencies

Internet access is required only to clone GitHub, not as a modeling input.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/fds-pokemon")
REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise ValueError(f"Not a Git repository: {REPO_DIR}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)


## Locate official competition files and generate all three submissions

In [ ]:
train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl; found {len(train_files)}")
DATA_DIR = train_files[0].parent
for name in ("test.jsonl", "type_chart_gen2.csv", "sample_submission.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUTPUT_DIR = Path("/kaggle/working/submissions")
subprocess.run([
    sys.executable, str(REPO_DIR / "run.py"),
    "--data", str(DATA_DIR), "--output-dir", str(OUTPUT_DIR)
], cwd=REPO_DIR, check=True)
print("Output directory:", OUTPUT_DIR, "Commit:", COMMIT)


In [ ]:
import hashlib
import pandas as pd

EXPECTED_FILES = (
    "submission_1_offense.csv",
    "submission_2_full.csv",
    "submission_3_consensus.csv",
)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
for filename in EXPECTED_FILES:
    path = OUTPUT_DIR / filename
    submission = pd.read_csv(path)
    assert submission.shape == sample.shape, filename
    assert submission.columns.equals(sample.columns), filename
    assert submission.battle_id.equals(sample.battle_id), filename
    assert submission.player_won.isin((0, 1)).all(), filename
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{filename}: {len(submission)} rows; sha256={digest}")
    display(submission.head(3))


All three selected submissions are saved in `/kaggle/working/submissions/`. Download them individually or attach them as notebook outputs. No test labels or previously scored submissions are used to create the predictions.